![logo](../../.././docs/images/Logo_Destination_Earth_Colours.png)

# Solution 4: Feature Extraction by Country - Climate DT

In this exercise, you'll extract data for a specific country from the Climate DT using polygon-based feature extraction. This solution uses Italy as an example.

## Authentication

In [ ]:
%%capture cap
%run ../../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [1]:
import earthkit.data
import earthkit.plots
import earthkit.geo.cartography
import os

## Live Request Toggle

In [2]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

True

## Select Your Country

In [3]:
countries = ["Italy"]

shapes = earthkit.geo.cartography.country_polygons(countries, resolution=50e6)
print(f"Extracted {len(shapes)} polygon(s) for {countries}")

Extracted 8 polygon(s) for ['Italy']


## Build the Feature Extraction Request

The `feature` key tells Polytope to extract data only within the country polygon.

In [4]:
request = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "experiment": "ssp3-7.0",
    "generation": "2",
    "levtype": "sfc",
    "date": "20400102",
    "model": "ifs-nemo",
    "expver": "0001",
    "param": "167",
    "realization": "1",
    "resolution": "high",
    "stream": "clte",
    "type": "fc",
    "time": "1200",
    "feature": {
        "type": "polygon",
        "shape": shapes,
    },
}

## Retrieve the Data

In [ ]:
if LIVE_REQUEST:
    data = earthkit.data.from_source(
        "polytope", "destination-earth", request,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
else:
    data = earthkit.data.from_source("file", "../../climate-dt/data/climate-dt-fe-country-training.grib")

## Convert to xarray

In [6]:
ds = data.to_xarray()
ds

<xarray.Dataset> Size: 297kB
Dimensions:    (datetimes: 1, number: 1, steps: 1, points: 7416)
Coordinates:
  * datetimes  (datetimes) <U20 80B '2040-01-02 12:00:00Z'
  * number     (number) int64 8B 0
  * steps      (steps) int64 8B 0
  * points     (points) int64 59kB 0 1 2 3 4 5 ... 7410 7411 7412 7413 7414 7415
    latitude   (points) float64 59kB 36.7 36.7 36.75 36.75 ... 47.01 47.01 47.06
    longitude  (points) float64 59kB 14.99 15.07 14.68 ... 12.0 12.09 12.11
    levelist   (points) float64 59kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Data variables:
    2t         (datetimes, number, steps, points) float64 59kB 284.5 ... 269.3
Attributes: (12/15)
    activity:     projections
    class:        d1
    dataset:      climate-dt
    experiment:   ssp3-7.0
    expver:       0001
    generation:   2
    ...           ...
    resolution:   high
    stream:       clte
    type:         fc
    number:       0
    step:         0
    date:         2040-01-02 12:00:00Z

## Plot Your Country

In [ ]:
chart = earthkit.plots.Map(domain=["Italy"])

chart.point_cloud(ds["2t"], units="celsius")
chart.coastlines()
chart.borders()
chart.gridlines()

chart.title("2m Temperature for Italy")

chart.legend()
chart.show()

## Bonus Exercise: 10m U Wind

Here's the same request with parameter 165 (10m U wind component).

In [ ]:
request_wind = request.copy()
request_wind["param"] = "165"

if LIVE_REQUEST:
    data_wind = earthkit.data.from_source(
        "polytope", "destination-earth", request_wind,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
    ds_wind = data_wind.to_xarray()

    chart = earthkit.plots.Map(domain=["Italy"])
    chart.point_cloud(ds_wind["10u"], units="m/s")
    chart.coastlines()
    chart.borders()
    chart.gridlines()
    chart.title("10m U Wind for Italy")
    chart.legend()
    chart.show()